# Phase 4 · Transfer Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MartinAlani/project_deep_learning_week_7/blob/main/notebooks/04_transfer_learning.ipynb)

**Goals of this notebook**
1. Compare **three pre-trained models** (ImageNet weights) as frozen feature extractors: **MobileNetV2**, **ResNet50V2** and **EfficientNetB0**.
2. Pick the best one using the **validation** set.
3. **Fine-tune** the winner to squeeze out extra accuracy.
4. Evaluate it on the **test set** and compare it with our custom CNN from Phase 2.

**Plan**
```
Stage 1 · Feature extraction   →  3 models, base frozen, only the new head is trained
Comparison                     →  choose the best model by validation accuracy
Stage 2 · Fine-tuning          →  unfreeze the winner's base, train with a tiny learning rate
Final evaluation               →  test set + comparison with the custom CNN
```

> 💡 **In Colab:** *Runtime → Change runtime type → T4 GPU* (required). The whole notebook takes roughly **1 hour**. Every model is saved to Google Drive as soon as it is trained, so if Colab disconnects you can re-run the notebook and finished models are **skipped** automatically.

## Why these three models (and not VGG16)?

| Model | Year | Params | ImageNet top-1 | Key idea |
|---|---|---|---|---|
| **MobileNetV2** | 2018 | 3.5 M | ~72% | Lightweight, built for mobile devices → fast inference |
| **ResNet50V2** | 2016 | 25.6 M | ~76% | Residual ("skip") connections allow very deep networks |
| **EfficientNetB0** | 2019 | 5.3 M | ~77% | Scales depth, width and resolution together → best accuracy per parameter |
| ~~VGG16~~ | 2014 | 138 M | ~71% | Discarded: 26× more parameters than EfficientNetB0, slow on a free GPU, >500 MB file (bad for deployment) and lower ImageNet accuracy |

We use **ResNet50V2** (the improved version of ResNet50) because it uses the same simple input scaling as MobileNetV2, which keeps the preprocessing inside the model clean.

## 1. Setup: imports, Google Drive and mixed precision

**Mixed precision** runs most computations in 16-bit floats instead of 32-bit. On the T4 GPU this makes training roughly **2× faster** with no loss in accuracy. The last layer stays in 32-bit for numerical stability.

In [ ]:
import os
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             precision_recall_fscore_support)

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow version:", tf.__version__)
print("GPU available:", gpus)

if gpus:
    keras.mixed_precision.set_global_policy("mixed_float16")
print("Precision policy:", keras.mixed_precision.global_policy().name)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/project_deep_learning_week_7"
except ImportError:
    PROJECT_DIR = ".."

MODELS_DIR = os.path.join(PROJECT_DIR, "models")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Models folder: ", MODELS_DIR)
print("Results folder:", RESULTS_DIR)

## 2. Settings

All the hyper-parameters of this phase in one place.

- `IMG_SIZE = 128`: CIFAR-10 images (32×32) are **upscaled** to 128×128. ImageNet models expect large images; at 32×32 the repeated pooling would shrink the image to almost nothing.
- `RETRAIN = False`: models already saved on Drive are loaded instead of retrained. Set it to `True` to train everything again.

In [ ]:
CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

IMG_SIZE = 128             # images are upscaled from 32x32 to this size
BATCH_SIZE = 64
WEIGHTS = "imagenet"       # pre-trained weights

STAGE1_EPOCHS = 10         # feature extraction (frozen base)
STAGE1_LR = 1e-3
FINE_TUNE_EPOCHS = 15      # fine-tuning (unfrozen base)
FINE_TUNE_LR = 1e-5        # 100x smaller: only small adjustments

RETRAIN = False            # True = ignore models already saved on Drive

## 3. Data (same split as every other phase)

Same `SEED` → identical train / validation / test sets, so the comparison with the custom CNN is fair.

In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train_full, y_test = y_train_full.flatten(), y_test.flatten()

x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full, test_size=5000, stratify=y_train_full, random_state=SEED
)

def make_dataset(x, y, training=False):
    """Turn NumPy arrays into a batched, prefetched tf.data.Dataset."""
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(buffer_size=len(x), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(x_train, y_train, training=True)
val_ds = make_dataset(x_val, y_val)
test_ds = make_dataset(x_test, y_test)

print("Train:", x_train.shape, "| Validation:", x_val.shape, "| Test:", x_test.shape)

## 4. Building a transfer-learning model

Every model follows the same structure:

```
Input 32×32×3 (raw pixels 0-255)
 ├─ Data augmentation          (same as Phase 2, active only during training)
 ├─ Resizing → 128×128         (upscale to a size ImageNet models can work with)
 ├─ Model-specific scaling     (the normalization each model was trained with)
 ├─ Pre-trained base 🧊        (ImageNet weights, include_top=False, FROZEN)
 ├─ GlobalAveragePooling2D     (one number per feature map)
 ├─ Dropout 0.3
 └─ Dense(10) + Softmax 🔥     (new head, the only trainable part in stage 1)
```

- `include_top=False` removes the original 1,000-class ImageNet classifier.
- `base.trainable = False` **freezes** the pre-trained weights.
- Resizing and scaling live **inside** the model, so the deployed app can still send raw 32×32 images.
- **GlobalAveragePooling2D** averages each feature map into one number. It has far fewer parameters than `Flatten` + a big `Dense` layer, which reduces overfitting.

In [ ]:
# Each candidate: its Keras constructor and the input scaling it was trained with
CANDIDATES = {
    "mobilenet_v2":    {"builder": keras.applications.MobileNetV2,    "scaling": "minus1_to_1"},
    "resnet50_v2":     {"builder": keras.applications.ResNet50V2,     "scaling": "minus1_to_1"},
    "efficientnet_b0": {"builder": keras.applications.EfficientNetB0, "scaling": "built_in"},
}


def scaling_layer(kind):
    """Return the input-scaling layer a pre-trained model expects.

    - "minus1_to_1": pixels [0, 255] → [-1, 1] (MobileNetV2, ResNet50V2)
    - "built_in": EfficientNet normalizes internally and expects raw [0, 255] pixels
    """
    if kind == "minus1_to_1":
        return layers.Rescaling(1.0 / 127.5, offset=-1.0, name="scaling")
    return layers.Identity(name="scaling")


def build_transfer_model(name):
    """Build a frozen pre-trained base + new classification head for CIFAR-10.

    Args:
        name: key of the CANDIDATES dictionary.

    Returns:
        (model, base): the full Keras model and its pre-trained base.
    """
    cfg = CANDIDATES[name]
    base = cfg["builder"](include_top=False, weights=WEIGHTS,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False   # 🧊 freeze all pre-trained weights

    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomZoom(0.1),
    ], name="data_augmentation")

    inputs = keras.Input(shape=(32, 32, 3), name="image")
    x = data_augmentation(inputs)
    x = layers.Resizing(IMG_SIZE, IMG_SIZE, name="resize")(x)
    x = scaling_layer(cfg["scaling"])(x)
    x = base(x, training=False)          # keep BatchNorm layers in inference mode
    x = layers.GlobalAveragePooling2D(name="global_avg_pool")(x)
    x = layers.Dropout(0.3, name="head_dropout")(x)
    outputs = layers.Dense(len(CLASS_NAMES), activation="softmax",
                           dtype="float32", name="predictions")(x)
    return keras.Model(inputs, outputs, name=f"tl_{name}"), base


def get_base(model):
    """Find the pre-trained base inside a (possibly reloaded) transfer model."""
    return next(layer for layer in model.layers
                if isinstance(layer, keras.Model) and not isinstance(layer, keras.Sequential))


# Quick look at one of them
demo_model, demo_base = build_transfer_model("efficientnet_b0")
demo_model.summary()
print(f"Base layers: {len(demo_base.layers)} | "
      f"trainable params: {sum(int(np.prod(w.shape)) for w in demo_model.trainable_weights):,} of "
      f"{demo_model.count_params():,}")
del demo_model, demo_base

Notice in the summary: the base has millions of **non-trainable** parameters (frozen), while only the last `Dense` layer (12,810 params for EfficientNetB0) is trainable in stage 1. That's why this stage is fast.

## 5. Stage 1 · Feature extraction 🧊

We train the three candidates under **identical conditions**: same data, same head, Adam with learning rate 0.001, up to 10 epochs, early stopping on validation loss (patience 3).

Each trained model and its results are saved to Drive immediately. With `RETRAIN = False`, models found on Drive are loaded instead of trained again.

In [ ]:
def results_path(name, stage):
    return os.path.join(RESULTS_DIR, f"tl_{name}_{stage}.json")


def model_path(name, stage):
    return os.path.join(MODELS_DIR, f"tl_{name}_{stage}.keras")


def train_stage1(name):
    """Train one candidate with a frozen base; save the model and its results to Drive."""
    if not RETRAIN and os.path.exists(results_path(name, "stage1")) and os.path.exists(model_path(name, "stage1")):
        print(f"⏭️  {name}: already trained, loading results from Drive")
        with open(results_path(name, "stage1")) as f:
            return json.load(f)

    print(f"\n🚀 Training {name} (stage 1: frozen base)")
    keras.utils.set_random_seed(SEED)
    model, _ = build_transfer_model(name)
    model.compile(optimizer=keras.optimizers.Adam(STAGE1_LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    start = time.time()
    history = model.fit(
        train_ds, validation_data=val_ds, epochs=STAGE1_EPOCHS,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                 restore_best_weights=True, verbose=1)],
    )
    minutes = (time.time() - start) / 60

    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    model.save(model_path(name, "stage1"))

    results = {
        "model": name,
        "stage": "stage1_feature_extraction",
        "total_params": int(model.count_params()),
        "trainable_params": int(sum(np.prod(w.shape) for w in model.trainable_weights)),
        "epochs_run": len(history.history["loss"]),
        "training_minutes": round(minutes, 1),
        "minutes_per_epoch": round(minutes / len(history.history["loss"]), 2),
        "val_accuracy": float(val_acc),
        "val_loss": float(val_loss),
        "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
    }
    with open(results_path(name, "stage1"), "w") as f:
        json.dump(results, f, indent=2)
    print(f"✅ {name}: val_accuracy = {val_acc:.4f} ({minutes:.1f} min)")
    return results

### 5.1 MobileNetV2

In [ ]:
stage1_results = {}
stage1_results["mobilenet_v2"] = train_stage1("mobilenet_v2")

### 5.2 ResNet50V2

In [ ]:
stage1_results["resnet50_v2"] = train_stage1("resnet50_v2")

### 5.3 EfficientNetB0

In [ ]:
stage1_results["efficientnet_b0"] = train_stage1("efficientnet_b0")

## 6. Comparison and choice of the best model

We choose the winner using **validation accuracy only**. The test set stays untouched until the very end.

In [ ]:
comparison = pd.DataFrame([
    {
        "model": r["model"],
        "val_accuracy": r["val_accuracy"],
        "val_loss": r["val_loss"],
        "total_params (M)": r["total_params"] / 1e6,
        "trainable_params": r["trainable_params"],
        "epochs": r["epochs_run"],
        "min/epoch": r["minutes_per_epoch"],
        "train_minutes": r["training_minutes"],
    }
    for r in stage1_results.values()
]).set_index("model").sort_values("val_accuracy", ascending=False)

BEST = comparison.index[0]
print(f"🏆 Best model after stage 1: {BEST}")
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

comparison["val_accuracy"].plot(kind="barh", ax=axes[0], color="steelblue")
axes[0].set_title("Stage 1 · validation accuracy")
axes[0].set_xlim(0, 1)
for i, v in enumerate(comparison["val_accuracy"]):
    axes[0].text(v + 0.01, i, f"{v:.2%}", va="center")

for name, r in stage1_results.items():
    axes[1].plot(range(1, len(r["history"]["val_accuracy"]) + 1),
                 r["history"]["val_accuracy"], marker="o", label=name)
axes[1].set_title("Stage 1 · validation accuracy per epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "tl_stage1_comparison.png"), dpi=120)
plt.show()

## 7. Stage 2 · Fine-tuning the best model 🔧

Now we **unfreeze** the pre-trained base of the winner and keep training with a **very small learning rate** (0.00001, 100× smaller than in stage 1).

- The base already knows a lot from ImageNet; we only want to **adjust** it to CIFAR-10. Large steps would destroy what it learned.
- **BatchNormalization layers stay frozen**: their statistics come from millions of ImageNet images, and updating them with small batches would make training unstable.
- Callbacks: early stopping (patience 4), ReduceLROnPlateau and a checkpoint on Drive.

In [ ]:
FINETUNED_PATH = model_path(BEST, "finetuned")

if not RETRAIN and os.path.exists(FINETUNED_PATH) and os.path.exists(results_path(BEST, "finetuned")):
    print(f"⏭️  {BEST}: fine-tuned model found on Drive, loading it")
    model = keras.models.load_model(FINETUNED_PATH)
    with open(results_path(BEST, "finetuned")) as f:
        finetune_results = json.load(f)
else:
    model = keras.models.load_model(model_path(BEST, "stage1"))
    base = get_base(model)

    base.trainable = True                       # 🔥 unfreeze the base...
    for layer in base.layers:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False             # ...but keep BatchNorm frozen

    model.compile(optimizer=keras.optimizers.Adam(FINE_TUNE_LR),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    n_trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    print(f"Trainable params now: {n_trainable:,} of {model.count_params():,}")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2,
                                          min_lr=1e-7, verbose=1),
        keras.callbacks.ModelCheckpoint(FINETUNED_PATH, monitor="val_loss",
                                        save_best_only=True, verbose=0),
    ]

    start = time.time()
    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=FINE_TUNE_EPOCHS, callbacks=callbacks)
    minutes = (time.time() - start) / 60

    val_loss, val_acc = model.evaluate(val_ds, verbose=0)
    model.save(FINETUNED_PATH)   # best weights (restored by early stopping)

    finetune_results = {
        "model": BEST,
        "stage": "stage2_fine_tuning",
        "learning_rate": FINE_TUNE_LR,
        "trainable_params": n_trainable,
        "epochs_run": len(history.history["loss"]),
        "training_minutes": round(minutes, 1),
        "val_accuracy": float(val_acc),
        "val_loss": float(val_loss),
        "history": {k: [float(v) for v in vals] for k, vals in history.history.items()},
    }
    with open(results_path(BEST, "finetuned"), "w") as f:
        json.dump(finetune_results, f, indent=2)

print(f"\nValidation accuracy · stage 1: {stage1_results[BEST]['val_accuracy']:.4f} "
      f"→ after fine-tuning: {finetune_results['val_accuracy']:.4f}")

### Learning curves: stage 1 + stage 2

The dashed line marks where fine-tuning starts.

In [ ]:
def plot_two_stages(stage1, stage2, title):
    """Plot accuracy and loss across feature extraction and fine-tuning."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    n1 = len(stage1["history"]["loss"])
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        for split, label in [(metric, "train"), (f"val_{metric}", "validation")]:
            values = stage1["history"][split] + stage2["history"][split]
            ax.plot(range(1, len(values) + 1), values, label=label)
        ax.axvline(n1 + 0.5, color="gray", linestyle="--", label="start fine-tuning")
        ax.set_title(f"{title} · {metric}")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(metric.capitalize())
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"tl_{BEST}_learning_curves.png"), dpi=120)
    plt.show()

plot_two_stages(stage1_results[BEST], finetune_results, BEST)

## 8. Final evaluation on the test set 🎓

For the first time in this notebook we use the **test set**, exactly once, to get an honest score for the fine-tuned model. We also measure how fast it predicts, which matters for the web app.

In [ ]:
start = time.time()
test_probs = model.predict(test_ds, verbose=0)
ms_per_image = (time.time() - start) / len(x_test) * 1000

y_pred = np.argmax(test_probs, axis=1)
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="macro", zero_division=0
)

print(f"Test accuracy:   {test_acc:.4f}")
print(f"Macro precision: {macro_p:.4f}")
print(f"Macro recall:    {macro_r:.4f}")
print(f"Macro F1-score:  {macro_f1:.4f}")
print(f"Prediction speed: {ms_per_image:.2f} ms per image (batch of {BATCH_SIZE})\n")
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_norm, annot=True, fmt=".0%", cmap="Blues", cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f"{BEST} (fine-tuned) · confusion matrix normalized by true class (test set)")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f"tl_{BEST}_confusion_matrix.png"), dpi=120)
plt.show()

## 9. Custom CNN vs transfer learning

We load the Phase 3 results of the custom CNN and compare both models on the **same test set**.

In [ ]:
with open(os.path.join(RESULTS_DIR, "custom_cnn_evaluation.json")) as f:
    cnn_eval = json.load(f)
with open(os.path.join(MODELS_DIR, "custom_cnn_history.json")) as f:
    cnn_train = json.load(f)

# Prediction speed of the custom CNN, measured the same way
cnn_model = keras.models.load_model(os.path.join(MODELS_DIR, "custom_cnn.keras"))
start = time.time()
cnn_model.predict(test_ds, verbose=0)
cnn_ms = (time.time() - start) / len(x_test) * 1000

def file_mb(path):
    return os.path.getsize(path) / 1e6

final_comparison = pd.DataFrame({
    "custom_cnn": {
        "test_accuracy": cnn_eval["test_accuracy"],
        "test_macro_f1": cnn_eval["test_macro_f1"],
        "params (M)": cnn_train["total_params"] / 1e6,
        "model size (MB)": file_mb(os.path.join(MODELS_DIR, "custom_cnn.keras")),
        "training minutes": cnn_train["training_minutes"],
        "ms / image": cnn_ms,
    },
    f"{BEST} (fine-tuned)": {
        "test_accuracy": float(test_acc),
        "test_macro_f1": float(macro_f1),
        "params (M)": model.count_params() / 1e6,
        "model size (MB)": file_mb(FINETUNED_PATH),
        "training minutes": stage1_results[BEST]["training_minutes"] + finetune_results["training_minutes"],
        "ms / image": ms_per_image,
    },
}).T
final_comparison.round(4)

In [ ]:
# Per-class F1: where does transfer learning help most?
_, _, f1_tl, _ = precision_recall_fscore_support(y_test, y_pred, zero_division=0)
f1_cnn = [cnn_eval["per_class"][c]["f1_score"] for c in CLASS_NAMES]

f1_compare = pd.DataFrame({"custom_cnn": f1_cnn, f"{BEST} (fine-tuned)": f1_tl}, index=CLASS_NAMES)
f1_compare["improvement"] = f1_compare.iloc[:, 1] - f1_compare.iloc[:, 0]

ax = f1_compare.iloc[:, :2].plot(kind="bar", figsize=(12, 5), width=0.8)
ax.set_title("F1-score per class · custom CNN vs transfer learning (test set)")
ax.set_ylabel("F1-score")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "cnn_vs_transfer_f1_per_class.png"), dpi=120)
plt.show()

f1_compare.sort_values("improvement", ascending=False).round(4)

## 10. Save the results

In [ ]:
tl_evaluation = {
    "model": BEST,
    "img_size": IMG_SIZE,
    "test_accuracy": float(test_acc),
    "test_loss": float(test_loss),
    "test_macro_precision": float(macro_p),
    "test_macro_recall": float(macro_r),
    "test_macro_f1": float(macro_f1),
    "ms_per_image": float(ms_per_image),
    "per_class": {c: {"f1_score": float(v)} for c, v in zip(CLASS_NAMES, f1_tl)},
    "confusion_matrix": cm.tolist(),
    "stage1_comparison": comparison.reset_index().to_dict(orient="records"),
}
with open(os.path.join(RESULTS_DIR, "transfer_learning_evaluation.json"), "w") as f:
    json.dump(tl_evaluation, f, indent=2, default=float)

print("Saved files:")
for folder in (MODELS_DIR, RESULTS_DIR):
    print(f"\n{folder}")
    for name in sorted(os.listdir(folder)):
        print(" -", name)

## ✅ Phase 4 summary

- **Stage 1** compared MobileNetV2, ResNet50V2 and EfficientNetB0 as frozen feature extractors under identical conditions; the best one was chosen on **validation** accuracy.
- **Stage 2** fine-tuned the winner with a 100× smaller learning rate and frozen BatchNorm layers.
- The final model was evaluated **once** on the test set and compared with the custom CNN on accuracy, F1, size and speed.

👉 **For the report:** save the stage-1 comparison chart, the two-stage learning curves, the confusion matrix, the comparison table and the per-class F1 chart.

**Next step → Phase 5:** deploy the best model in a Flask web app.